# Road Defect Detector — YOLO11n training (GPU)

Trains the detector used by the **Automated Road Health Assessment & Grievance Tool**.

| | |
|---|---|
| Datasets | `siva-ragampudi/pothole-vhmow-jwwzq` v1 + `siva-ragampudi/road-c013o-tlkm7` v1 |
| Images | 3,224 (train 2,309 / val 601 / test 314) |
| Classes | `0 = pothole`, `1 = crack` |
| Output | `best.pt` (~5.5 MB) → drop into `backend/weights/best.pt` |

**Baseline to beat** — the CPU-trained model this replaces, measured on the held-out test set:

| | mAP50 | mAP50-95 | precision | recall |
|---|---|---|---|---|
| 8-epoch CPU model | 0.520 | 0.220 | 0.602 | 0.502 |

On a T4 this runs 100 epochs at 640 px in roughly 35–50 minutes.

---
### Before you start
**Runtime → Change runtime type → T4 GPU**, then Runtime → Run all.

## 1 · Check the GPU
If this prints `cpu`, you didn't switch the runtime — go back and do it, or you'll wait hours instead of minutes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU - switch runtime to T4!'
%pip install -q ultralytics roboflow
import torch
print('torch', torch.__version__, '| device:', 'cuda' if torch.cuda.is_available() else 'cpu')
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then Run all again.'

## 2 · Download both datasets

Get your key from **app.roboflow.com → workspace Settings → API Keys**. It is only used to download.

These are the *forked* copies in your own workspace, each with a generated v1 — a fork has no version until one is generated, which is why the plain Universe slugs won't work here.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

rf = Roboflow(api_key=getpass('Roboflow API key: '))
ws = rf.workspace('siva-ragampudi')

ws.project('pothole-vhmow-jwwzq').version(1).download('yolov11', location='/content/datasets/pothole')
ws.project('road-c013o-tlkm7').version(1).download('yolov11', location='/content/datasets/road')
print('\ndownloaded')

## 3 · Fix the class collision  ⚠️ do not skip

The two datasets disagree about what each class id means:

| | class 0 | class 1 | class 2 |
|---|---|---|---|
| pothole (`nc:1`) | `pothole` | — | — |
| road (`nc:3`) | `Crack` | `crack` | `pothole` |

Class 0 means **pothole** in one and **Crack** in the other. Merging as-is silently relabels 472 cracks as potholes and leaves road's potholes on an out-of-range index. This cell rewrites road's labels to the unified scheme (`0=pothole`, `1=crack`) and is guarded so re-running can't double-apply it.

In [ ]:
import collections, pathlib

road = pathlib.Path('/content/datasets/road')
guard = road / '.remapped_to_unified'
REMAP = {0: 1, 1: 1, 2: 0}   # 0 Crack->crack, 1 crack->crack, 2 pothole->pothole

if guard.exists():
    print('already remapped - skipping')
else:
    before, after, n = collections.Counter(), collections.Counter(), 0
    for split in ('train', 'valid', 'test'):
        for lf in (road / split / 'labels').glob('*.txt'):
            out = []
            for line in lf.read_text().splitlines():
                p = line.split()
                if not p:
                    continue
                before[int(p[0])] += 1
                p[0] = str(REMAP[int(p[0])])
                after[int(p[0])] += 1
                out.append(' '.join(p))
            lf.write_text('\n'.join(out) + ('\n' if out else ''))
            n += 1
    guard.write_text('0(Crack)->1, 1(crack)->1, 2(pothole)->0\n')
    old = ['Crack', 'crack', 'pothole']
    print(f'rewrote {n} label files')
    print('BEFORE:', {old[k]: v for k, v in sorted(before.items())})
    print('AFTER :', {['pothole', 'crack'][k]: v for k, v in sorted(after.items())})
    print('\nexpect AFTER pothole=1739, crack=1675  (1203 + 472 merged)')

## 4 · Unified data.yaml
Multi-path lists let YOLO read both datasets in place — no image copying, no duplication.

In [ ]:
yaml_text = '''# Unified road-defect dataset. Class scheme is ours: 0 = pothole, 1 = crack.
# road/ labels were remapped in place by the previous cell.
path: /content/datasets
train: [pothole/train/images, road/train/images]
val:   [pothole/valid/images, road/valid/images]
test:  [pothole/test/images,  road/test/images]
nc: 2
names:
  0: pothole
  1: crack
'''
open('/content/road-health.yaml', 'w').write(yaml_text)

import pathlib
for s in ('train', 'valid', 'test'):
    p = len(list(pathlib.Path(f'/content/datasets/pothole/{s}/images').glob('*')))
    r = len(list(pathlib.Path(f'/content/datasets/road/{s}/images').glob('*')))
    print(f'{s:6s}: pothole {p:5d} + road {r:5d} = {p + r}')

## 5 · Train

Two deliberate choices worth understanding:

- **`imgsz=640`** — the CPU run used 416 purely because 640 was too slow. On a GPU there's no reason to compromise, and small distant cracks are exactly what low resolution loses. *If you train at 640, set `INFERENCE_IMAGE_SIZE=640` in the backend to match.*
- **`flipud=0.0`** — vertical flip is off on purpose. An upside-down road photo never occurs in the capture pipeline, so it only adds noise.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
results = model.train(
    data='/content/road-health.yaml',
    epochs=100,
    patience=20,
    imgsz=640,
    batch=32,
    device=0,
    project='/content/runs',
    name='road_yolo11n',
    seed=0,
    plots=True,
    flipud=0.0,
    fliplr=0.5,
    degrees=10,
    translate=0.1,
    scale=0.5,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    mosaic=1.0,
    close_mosaic=10,
)

## 6 · Evaluate on the held-out test set
The test split was never used for model selection, so this is the number to trust — and the one to compare against the 0.520 baseline.

In [ ]:
best = YOLO('/content/runs/road_yolo11n/weights/best.pt')
m = best.val(data='/content/road-health.yaml', split='test', imgsz=640, device=0)

print('\n=== TEST SET ===')
print(f'mAP50    : {m.box.map50:.4f}   (CPU baseline 0.520)')
print(f'mAP50-95 : {m.box.map:.4f}   (CPU baseline 0.220)')
print(f'precision: {m.box.mp:.4f}   (CPU baseline 0.602)')
print(f'recall   : {m.box.mr:.4f}   (CPU baseline 0.502)')
for i, name in enumerate(['pothole', 'crack']):
    print(f'  {name:8s} mAP50={m.box.ap50[i]:.4f}  P={m.box.p[i]:.4f}  R={m.box.r[i]:.4f}')

## 7 · Look at real predictions
Numbers hide failure modes. The CPU model scored a respectable 0.520 while confidently boxing thin cracks and calling them potholes — only visible by looking. Check that boxes sit on actual defects and the labels are right.

In [ ]:
import glob
from IPython.display import Image, display

imgs = sorted(glob.glob('/content/datasets/pothole/test/images/*.jpg'))[:4] + \
       sorted(glob.glob('/content/datasets/road/test/images/*.jpg'))[:4]
best.predict(imgs, imgsz=640, conf=0.25, save=True, project='/content/preds', name='check', exist_ok=True)
for f in sorted(glob.glob('/content/preds/check/*.jpg')):
    display(Image(filename=f, width=430))

## 8 · Strip and download
A raw checkpoint is ~21 MB because it carries optimizer + EMA state. Stripping leaves ~5.5 MB — small enough to commit, which matters because Render deploys the model straight from your repo.

In [ ]:
import shutil, pathlib
from ultralytics.utils.torch_utils import strip_optimizer
from google.colab import files

src = pathlib.Path('/content/runs/road_yolo11n/weights/best.pt')
out = pathlib.Path('/content/best.pt')
shutil.copy2(src, out)
print(f'before: {src.stat().st_size / 1e6:.1f} MB')
strip_optimizer(out)
print(f'after : {out.stat().st_size / 1e6:.1f} MB')

files.download('/content/best.pt')
files.download('/content/runs/road_yolo11n/results.png')
files.download('/content/runs/road_yolo11n/confusion_matrix_normalized.png')

## 9 · Install it in the backend

```bash
cd ~/Desktop/mini_project/backend
cp ~/Downloads/best.pt weights/best.pt

# trained at 640 in this notebook, so match it:
#   app/config.py   inference_image_size default -> 640
#   .env.example    INFERENCE_IMAGE_SIZE=640
#   render.yaml     INFERENCE_IMAGE_SIZE "640"

python3 -m pytest -q                 # 86 tests
uvicorn app.main:app --reload
curl http://localhost:8000/health    # model_loaded: true, classes [pothole, crack]
```

**Cost of 640 over 416:** inference measured 45.7 ms/image at 416 on CPU; 640 is ~2.4× the pixels, so expect roughly 110–150 ms on Render's free tier. Still fast. If that ever matters more than accuracy, retrain at 416 and change the three settings back.

`results.png` and `confusion_matrix_normalized.png` are worth keeping for your project report — the confusion matrix in particular will show whether cracks are still being called potholes.